In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

In [2]:
candidate_sites = pd.read_parquet("../../data/processed/retrieval/candidate_sites.parquet")
training_pairs = pd.read_parquet("../../data/processed/retrieval/training_pairs.parquet")
queries = pd.read_json("../../data/processed/retrieval/query_intents.jsonl", lines=True)

print("Candidates:", len(candidate_sites))
print("Training pairs:", len(training_pairs))
print("Queries:", len(queries))

Candidates: 49950
Training pairs: 12600
Queries: 21


In [3]:
print(candidate_sites.columns)
print(training_pairs.columns)
print(queries.columns)

Index(['RID', 'address', 'primary_zoning_code', 'primary_zoning_class',
       'zoning_band', 'mixed_zoning_flag', 'lot_size_proxy_sqm',
       'lot_size_band', 'heritage_flag', 'heritage_max_significance',
       'bushfire_flag', 'bushfire_risk_level', 'flood_flag',
       'primary_flood_class', 'distance_to_station_m', 'within_800m_catchment',
       'station_distance_band', 'station_access_score',
       'constraint_severity_band', 'top_strategy', 'top_strategy_score',
       'site_summary_text', 'candidate_text_debug', 'candidate_text_clean',
       'single_dwelling_rebuild_score', 'assembly_opportunity_score',
       'granny_flat_score', 'land_bank_hold_score',
       'townhouse_multi_dwelling_score', 'low_rise_apartment_score',
       'dual_occupancy_score'],
      dtype='object')
Index(['query_id', 'strategy', 'query_text', 'candidate_rid', 'candidate_text',
       'candidate_score', 'label', 'pair_type', 'address', 'top_strategy',
       'top_strategy_score', 'primary_zoning_co

In [4]:
TEXT_COL = "candidate_text_clean"

candidate_sites[TEXT_COL] = candidate_sites[TEXT_COL].fillna("").astype(str)
queries["text"] = queries["text"].fillna("").astype(str)

print(candidate_sites[[TEXT_COL]].head(3))
print(queries[["query_id", "strategy", "text"]].head(3))

                                candidate_text_clean
0  zoning_code R2 | zoning_band low_dev | lot_siz...
1  zoning_code R5 | zoning_band low_dev | lot_siz...
2  zoning_code R2 | zoning_band low_dev | lot_siz...
  query_id                 strategy  \
0     q001  single_dwelling_rebuild   
1     q002  single_dwelling_rebuild   
2     q003  single_dwelling_rebuild   

                                                text  
0  strategy single_dwelling_rebuild | zoning pref...  
1  strategy single_dwelling_rebuild | detached re...  
2  strategy single_dwelling_rebuild | standard re...  


In [5]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
candidate_texts = candidate_sites[TEXT_COL].tolist()
query_texts = queries["text"].tolist()

print(candidate_texts[:2])
print(query_texts[:2])

['zoning_code R2 | zoning_band low_dev | lot_size_band m | station_distance_band over_10km | constraint_severity low | mixed_zoning no | heritage no | flood no | bushfire_risk 0 | within_800m no', 'zoning_code R5 | zoning_band low_dev | lot_size_band xl | station_distance_band 2km_5km | constraint_severity high | mixed_zoning no | heritage no | flood no | bushfire_risk 3 | within_800m no']
['strategy single_dwelling_rebuild | zoning preference low_dev or mid_dev | lot size preference s m l | access preference not critical | avoid heritage flood severe bushfire', 'strategy single_dwelling_rebuild | detached residential rebuild | zoning low_dev mid_dev | moderate lot size | low constraint preferred']


In [7]:
candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

candidate_embeddings.shape

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

(49950, 384)

In [8]:
query_embeddings = model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_embeddings.shape

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(21, 384)

In [9]:
similarity_matrix = query_embeddings @ candidate_embeddings.T
similarity_matrix.shape

(21, 49950)

In [10]:
def retrieve_top_k(query_idx: int, k: int = 10) -> pd.DataFrame:
    sims = similarity_matrix[query_idx]
    top_idx = np.argsort(-sims)[:k]

    result = candidate_sites.iloc[top_idx].copy()
    result["similarity"] = sims[top_idx]
    result["query_id"] = queries.iloc[query_idx]["query_id"]
    result["query_text"] = queries.iloc[query_idx]["text"]
    result["strategy"] = queries.iloc[query_idx]["strategy"]
    return result

In [11]:
q_idx = 0
top10 = retrieve_top_k(q_idx, k=10)

display(
    top10[
        [
            "query_id",
            "strategy",
            "address",
            "primary_zoning_code",
            "lot_size_band",
            "constraint_severity_band",
            "station_distance_band",
            "top_strategy",
            "top_strategy_score",
            "similarity",
            TEXT_COL,
        ]
    ]
)

,query_id,strategy,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity,candidate_text_clean
46451,q001,single_dwelling_rebuild,1A GOLDSMITH STREET WOONGARRAH,R1,s,high,2km_5km,single_dwelling_rebuild,45.2,0.684092,zoning_code R1 | zoning_band mid_dev | lot_siz...
34260,q001,single_dwelling_rebuild,19/36-38 BUSACO ROAD MARSFIELD,R4,l,high,2km_5km,land_bank_hold,57.6,0.683765,zoning_code R4 | zoning_band high_dev | lot_si...
46426,q001,single_dwelling_rebuild,2/34 SCOTT STREET HARRINGTON,R1,l,high,over_10km,single_dwelling_rebuild,59.0,0.683753,zoning_code R1 | zoning_band mid_dev | lot_siz...
42791,q001,single_dwelling_rebuild,41 MACQUARIE DRIVE BURRILL LAKE,R1,l,high,over_10km,single_dwelling_rebuild,59.0,0.683753,zoning_code R1 | zoning_band mid_dev | lot_siz...
34484,q001,single_dwelling_rebuild,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.683339,zoning_code R1 | zoning_band mid_dev | lot_siz...
32703,q001,single_dwelling_rebuild,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.683339,zoning_code R1 | zoning_band mid_dev | lot_siz...
24021,q001,single_dwelling_rebuild,10 ADELAIDE CLOSE WINGHAM,R1,l,high,over_10km,single_dwelling_rebuild,54.0,0.683283,zoning_code R1 | zoning_band mid_dev | lot_siz...
28950,q001,single_dwelling_rebuild,13 ALLUMBA CLOSE TAREE,R1,l,high,over_10km,single_dwelling_rebuild,54.0,0.683283,zoning_code R1 | zoning_band mid_dev | lot_siz...
34120,q001,single_dwelling_rebuild,602/72 DONNISON STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,61.0,0.682759,zoning_code R1 | zoning_band mid_dev | lot_siz...
41749,q001,single_dwelling_rebuild,1 FEDERATION BOULEVARD WARNERVALE,R1,m,high,within_800m,single_dwelling_rebuild,60.0,0.682759,zoning_code R1 | zoning_band mid_dev | lot_siz...


In [12]:
for q_idx in range(min(7, len(queries))):
    q = queries.iloc[q_idx]
    print("\n" + "=" * 80)
    print("QUERY:", q["query_id"], "|", q["strategy"])
    print(q["text"])

    topk = retrieve_top_k(q_idx, k=5)
    display(
        topk[
            [
                "address",
                "primary_zoning_code",
                "lot_size_band",
                "constraint_severity_band",
                "station_distance_band",
                "top_strategy",
                "top_strategy_score",
                "similarity",
            ]
        ]
    )


QUERY: q001 | single_dwelling_rebuild
strategy single_dwelling_rebuild | zoning preference low_dev or mid_dev | lot size preference s m l | access preference not critical | avoid heritage flood severe bushfire


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
46451,1A GOLDSMITH STREET WOONGARRAH,R1,s,high,2km_5km,single_dwelling_rebuild,45.2,0.684092
34260,19/36-38 BUSACO ROAD MARSFIELD,R4,l,high,2km_5km,land_bank_hold,57.6,0.683765
46426,2/34 SCOTT STREET HARRINGTON,R1,l,high,over_10km,single_dwelling_rebuild,59.0,0.683753
42791,41 MACQUARIE DRIVE BURRILL LAKE,R1,l,high,over_10km,single_dwelling_rebuild,59.0,0.683753
34484,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.683339



QUERY: q002 | single_dwelling_rebuild
strategy single_dwelling_rebuild | detached residential rebuild | zoning low_dev mid_dev | moderate lot size | low constraint preferred


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
34484,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.597960
32703,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.597960
5658,22 JASMYNE STREET LISMORE,R1,m,high,within_800m,dual_occupancy,61.0,0.597655
41749,1 FEDERATION BOULEVARD WARNERVALE,R1,m,high,within_800m,single_dwelling_rebuild,60.0,0.597655
34120,602/72 DONNISON STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,61.0,0.597655



QUERY: q003 | single_dwelling_rebuild
strategy single_dwelling_rebuild | standard residential redevelopment | prefers R1 R2 R5 RU5 style context | avoid strong planning constraints


...

,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
32329,10/79-83 FAUNCE STREET WEST GOSFORD,R1,l,high,within_800m,dual_occupancy,66.0,0.543306
34484,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.541376
32703,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.541376
37708,13/77 FAUNCE STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,66.0,0.540468
2163,5/6 MARGIN STREET GOSFORD,R1,m,high,within_800m,single_dwelling_rebuild,65.0,0.540468



QUERY: q006 | assembly_opportunity
strategy assembly_opportunity | land aggregation potential | mixed zoning or larger site | strong station access preferred | not low density


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
32703,803/20 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.636351
34484,309/8 KENDALL STREET GOSFORD,R1,l,high,within_800m,dual_occupancy,61.0,0.636351
32329,10/79-83 FAUNCE STREET WEST GOSFORD,R1,l,high,within_800m,dual_occupancy,66.0,0.634485
5658,22 JASMYNE STREET LISMORE,R1,m,high,within_800m,dual_occupancy,61.0,0.634180
34120,602/72 DONNISON STREET WEST GOSFORD,R1,m,high,within_800m,dual_occupancy,61.0,0.634180



QUERY: q007 | granny_flat
strategy granny_flat | zoning preference low_dev | lot size preference m l xl | lower intensity residential use | avoid heritage flood severe bushfire


,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,top_strategy_score,similarity
28676,11 CATHERINE PARK DRIVE ORAN PARK,R2,m,high,5km_10km,single_dwelling_rebuild,41.0,0.667053
46742,32 RENWICK STREET CATHERINE FIELD,R2,xs,high,5km_10km,single_dwelling_rebuild,21.0,0.666962
46709,121-123 CANDLING ROAD LEPPINGTON,R2,m,high,2km_5km,dual_occupancy,42.3,0.665010
39498,513 DENHAM COURT ROAD LEPPINGTON,R2,s,high,2km_5km,single_dwelling_rebuild,32.2,0.662815
36224,39 FARVIEW DRIVE DENHAM COURT,R2,s,high,2km_5km,single_dwelling_rebuild,32.2,0.662815


In [13]:
def top_k_strategy_match_rate(query_idx: int, k: int = 10) -> float:
    strategy = queries.iloc[query_idx]["strategy"]
    topk = retrieve_top_k(query_idx, k=k)
    return (topk["top_strategy"] == strategy).mean()

match_rows = []
for i in range(len(queries)):
    match_rows.append(
        {
            "query_id": queries.iloc[i]["query_id"],
            "strategy": queries.iloc[i]["strategy"],
            "top10_match_rate": top_k_strategy_match_rate(i, k=10),
            "top20_match_rate": top_k_strategy_match_rate(i, k=20),
        }
    )

match_df = pd.DataFrame(match_rows)
match_df

,query_id,strategy,top10_match_rate,top20_match_rate
0,q001,single_dwelling_rebuild,0.6,0.30
1,q002,single_dwelling_rebuild,0.1,0.05
2,q003,single_dwelling_rebuild,0.7,0.85
3,q004,assembly_opportunity,0.0,0.00
4,q005,assembly_opportunity,0.0,0.00
5,q006,assembly_opportunity,0.0,0.00
6,q007,granny_flat,0.0,0.00
7,q008,granny_flat,0.0,0.00
8,q009,granny_flat,0.0,0.00
9,q010,land_bank_hold,0.1,0.10


In [14]:
match_df[["top10_match_rate", "top20_match_rate"]].mean()

top10_match_rate    0.176190
top20_match_rate    0.161905
dtype: float64

In [15]:
def score_col_for_strategy(strategy: str) -> str:
    return f"{strategy}_score"

score_check_rows = []
for i in range(len(queries)):
    strategy = queries.iloc[i]["strategy"]
    score_col = score_col_for_strategy(strategy)
    topk = retrieve_top_k(i, k=20)

    score_check_rows.append(
        {
            "query_id": queries.iloc[i]["query_id"],
            "strategy": strategy,
            "top20_mean_score": topk[score_col].mean(),
            "top20_median_score": topk[score_col].median(),
            "top20_high_score_rate": (topk[score_col] >= 70).mean(),
        }
    )

score_check_df = pd.DataFrame(score_check_rows)
score_check_df

,query_id,strategy,top20_mean_score,top20_median_score,top20_high_score_rate
0,q001,single_dwelling_rebuild,48.610,43.7,0.00
1,q002,single_dwelling_rebuild,51.850,48.0,0.00
2,q003,single_dwelling_rebuild,62.080,67.2,0.45
3,q004,assembly_opportunity,44.330,43.0,0.05
4,q005,assembly_opportunity,47.650,41.5,0.25
5,q006,assembly_opportunity,46.750,45.5,0.00
6,q007,granny_flat,32.535,37.5,0.00
7,q008,granny_flat,69.100,77.5,0.55
8,q009,granny_flat,38.500,38.5,0.00
9,q010,land_bank_hold,53.850,53.6,0.00


In [16]:
score_check_df[["top20_mean_score", "top20_median_score", "top20_high_score_rate"]].mean()

top20_mean_score         52.603333
top20_median_score       53.895238
top20_high_score_rate     0.228571
dtype: float64

In [17]:
all_topk = []
for i in range(len(queries)):
    topk = retrieve_top_k(i, k=50).copy()
    all_topk.append(topk)

baseline_retrieval = pd.concat(all_topk, ignore_index=True)

Path("../../data/processed/retrieval").mkdir(parents=True, exist_ok=True)
baseline_retrieval.to_parquet(
    "../../data/processed/retrieval/embedding_baseline_clean_top50.parquet",
    index=False,
)

print(len(baseline_retrieval))
baseline_retrieval.head()

1050


,RID,address,primary_zoning_code,primary_zoning_class,zoning_band,mixed_zoning_flag,lot_size_proxy_sqm,lot_size_band,heritage_flag,heritage_max_significance,...,assembly_opportunity_score,granny_flat_score,land_bank_hold_score,townhouse_multi_dwelling_score,low_rise_apartment_score,dual_occupancy_score,similarity,query_id,query_text,strategy
0,7024072,1A GOLDSMITH STREET WOONGARRAH,R1,General Residential,mid_dev,0,684.790576,s,0,None,...,8.6,33.5,12.6,8.0,0.0,23.8,0.684092,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
1,4399564,19/36-38 BUSACO ROAD MARSFIELD,R4,High Density Residential,high_dev,0,4291.706099,l,0,None,...,53.6,14.5,57.6,38.0,32.5,14.8,0.683765,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
2,7021362,2/34 SCOTT STREET HARRINGTON,R1,General Residential,mid_dev,1,2520.211197,l,0,None,...,30.0,57.0,44.0,40.0,5.0,57.0,0.683753,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
3,6344062,41 MACQUARIE DRIVE BURRILL LAKE,R1,General Residential,mid_dev,1,4067.895233,l,0,None,...,40.0,57.0,44.0,40.0,5.0,57.0,0.683753,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild
4,4473940,309/8 KENDALL STREET GOSFORD,R1,General Residential,mid_dev,0,3039.426763,l,0,None,...,43.0,59.5,57.0,50.0,22.5,61.0,0.683339,q001,strategy single_dwelling_rebuild | zoning pref...,single_dwelling_rebuild


In [18]:
strategy = "low_rise_apartment"
q_idx = queries.index[queries["strategy"] == strategy][0]

topk = retrieve_top_k(q_idx, k=20)
display(
    topk[
        [
            "address",
            "primary_zoning_code",
            "lot_size_proxy_sqm",
            "within_800m_catchment",
            "heritage_flag",
            "bushfire_risk_level",
            "flood_flag",
            "top_strategy",
            "top_strategy_score",
            "low_rise_apartment_score",
            "similarity",
            TEXT_COL,
        ]
    ]
)

,address,primary_zoning_code,lot_size_proxy_sqm,within_800m_catchment,heritage_flag,bushfire_risk_level,flood_flag,top_strategy,top_strategy_score,low_rise_apartment_score,similarity,candidate_text_clean
34241,126 CRIMEA ROAD MARSFIELD,R4,4785.795573,0,0,3,0,land_bank_hold,68.6,45.0,0.734700,zoning_code R4 | zoning_band high_dev | lot_si...
24974,9-17 ROBERTSON STREET SUTHERLAND,R4,4760.679506,1,0,0,0,assembly_opportunity,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
19370,7/19 ENGLISH STREET KOGARAH,R4,2520.384608,1,0,0,0,land_bank_hold,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
43886,16/16-18 BOUVARDIA STREET ASQUITH,R4,2631.507746,1,0,0,0,land_bank_hold,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
25175,48R CHARLES PLACE JANNALI,R4,2481.580006,1,0,0,0,land_bank_hold,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
4533,14/6 CARR STREET WAVERTON,R4,2824.849524,1,0,0,0,land_bank_hold,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
25238,16/29-33 KERRS ROAD LIDCOMBE,R4,2297.605701,1,0,0,0,land_bank_hold,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
38417,23/3-5 FREEMAN ROAD CHATSWOOD,R4,3230.047198,1,0,0,0,land_bank_hold,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
38418,7/3-5 FREEMAN ROAD CHATSWOOD,R4,3230.047198,1,0,0,0,land_bank_hold,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...
43602,501/8A ALLAWAH STREET BLACKTOWN,R4,4580.007216,1,0,0,0,assembly_opportunity,93.0,77.5,0.734221,zoning_code R4 | zoning_band high_dev | lot_si...


In [19]:
print(match_df)
print(score_check_df)
print(match_df[["top10_match_rate", "top20_match_rate"]].mean())
print(score_check_df[["top20_mean_score", "top20_median_score", "top20_high_score_rate"]].mean())

   query_id                  strategy  top10_match_rate  top20_match_rate
0      q001   single_dwelling_rebuild               0.6              0.30
1      q002   single_dwelling_rebuild               0.1              0.05
2      q003   single_dwelling_rebuild               0.7              0.85
3      q004      assembly_opportunity               0.0              0.00
4      q005      assembly_opportunity               0.0              0.00
5      q006      assembly_opportunity               0.0              0.00
6      q007               granny_flat               0.0              0.00
7      q008               granny_flat               0.0              0.00
8      q009               granny_flat               0.0              0.00
9      q010            land_bank_hold               0.1              0.10
10     q011            land_bank_hold               0.7              0.70
11     q012            land_bank_hold               0.6              0.60
12     q013  townhouse_multi_dwelling 